# Role intervention results

Layer-wise effects of agent/patient edits in Pythia 6.9B (`step143000`). This notebook reads the completed run's baseline and summary CSVs; it needs no model weights or GPU.

The fixed question is **Who is performing the action?** The agent is the original answer and the patient is the counterfactual answer.

- Score: `patient_logit - agent_logit`; positive values favor the patient.
- Change: intervention score minus baseline score; positive values shift toward the patient.
- Blue lines edit the agent token; orange lines edit the patient token. Solid lines use pooled role means, and dashed lines use means for the sentence's voice.

Panels keep `test` and `gen_test` separate, and distinguish active from passive voice. All curves are descriptive averages of this fixed corpus, with no uncertainty intervals or significance claims.

In [ ]:
%matplotlib inline
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebook":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Run from the repository root or notebook directory"

RUN = ROOT / "ac-tpr-cache/interventions/pythia-6.9b-step143000-generated-agent-tl3"
FIGURES = ROOT / "notebook/figures/role_interventions_tl3"
metadata = json.loads((RUN / "metadata.json").read_text())
assert metadata["schema_version"] == 3
baseline = pd.read_csv(RUN / "baseline.csv")
summary = pd.read_csv(RUN / "summary.csv")

SPLITS = ["test", "gen_test"]
VOICES = ["active", "passive"]
GROUPS = ["split", "voice"]
CONDITIONS = ["mean_type", "edited_role"]
KEYS = GROUPS + ["layer"] + CONDITIONS
LAYERS = metadata["layer_indices"]
assert metadata["limit"] is None, "Use the completed full run, not a smoke run"
assert len(baseline) == metadata["baseline_rows"]
assert len(summary) == metadata["summary_rows"]
assert not baseline.duplicated(["pair_id", "voice"]).any()
assert not summary.duplicated(KEYS).any()
assert np.isfinite(summary.select_dtypes("number")).all().all()
expected_keys = pd.MultiIndex.from_product(
    [SPLITS, VOICES, LAYERS, ["pooled", "by_voice"], ["agent", "patient"]],
    names=KEYS,
)
assert summary.set_index(KEYS).index.sort_values().equals(expected_keys.sort_values())
assert (summary.loc[summary.layer == LAYERS[-1], "mean_logit_diff_change"] == 0).all()

print(f"Model: {metadata['model']['name']} / {metadata['model']['revision']}")
print(f"Run: {RUN.resolve()}")
print(f"{len(baseline):,} sentences; {metadata['result_rows']:,} interventions; {len(summary)} groups")
print("Controls:", metadata["controls"])

## Baseline answer preferences

Preference is a strict comparison of the two noun logits, not unrestricted answer-generation accuracy. Ties favor neither answer. The baseline table supplies the denominators for the conditional flip plot below.

In [ ]:
baseline_stats = baseline.groupby(GROUPS).agg(
    sentences=("baseline_logit_diff", "size"),
    mean_logit_diff=("baseline_logit_diff", "mean"),
    agent_wins=("baseline_logit_diff", lambda values: (values < 0).sum()),
    patient_wins=("baseline_logit_diff", lambda values: (values > 0).sum()),
    ties=("baseline_logit_diff", lambda values: (values == 0).sum()),
).reindex(pd.MultiIndex.from_product([SPLITS, VOICES], names=GROUPS))
assert baseline_stats.notna().all().all()
for role in ("agent", "patient"):
    baseline_stats[f"{role}_preference"] = (
        baseline_stats[f"{role}_wins"] / baseline_stats.sentences
    )
baseline_stats["tie_rate"] = baseline_stats.ties / baseline_stats.sentences

# Check summary baselines against the individual sentence scores.
for group, rows in summary.groupby(GROUPS):
    stats = baseline_stats.loc[group]
    assert (rows["count"] == stats.sentences).all()
    for summary_name, baseline_name in (
        ("baseline_mean_logit_diff", "mean_logit_diff"),
        ("baseline_agent_preference_rate", "agent_preference"),
        ("baseline_patient_preference_rate", "patient_preference"),
    ):
        np.testing.assert_allclose(rows[summary_name], stats[baseline_name], atol=1e-12)
np.testing.assert_allclose(
    summary.intervened_mean_logit_diff - summary.baseline_mean_logit_diff,
    summary.mean_logit_diff_change,
    atol=1e-12,
)
assert (summary.baseline_agent_preference_rate > 0).all()
summary["conditional_flip_rate"] = (
    summary.agent_to_patient_flip_rate / summary.baseline_agent_preference_rate
)
assert summary.conditional_flip_rate.between(0, 1).all()
display(baseline_stats.round(4))

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94a3b8",
    "axes.labelcolor": "#334155",
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "figure.dpi": 130,
    "savefig.dpi": 200,
    "pdf.fonttype": 42,
})
COLORS = {"agent": "#0072B2", "patient": "#D55E00"}
STYLES = {"pooled": "-", "by_voice": "--"}
CONDITION_ORDER = [
    ("pooled", "agent"), ("pooled", "patient"),
    ("by_voice", "agent"), ("by_voice", "patient"),
]
CONDITION_LABELS = {
    (means, role): f"{role.capitalize()} edit / {'pooled' if means == 'pooled' else 'by voice'}"
    for means, role in CONDITION_ORDER
}
FIGURES.mkdir(parents=True, exist_ok=True)

def save_figure(fig, name):
    """Export the displayed figure as a raster preview and a vector PDF."""
    for extension in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{extension}", bbox_inches="tight")
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.7), sharey=True)
for ax, split in zip(axes, SPLITS):
    stats = baseline_stats.loc[split].reindex(VOICES)
    bottom = np.zeros(len(VOICES))
    for column, label, color in [
        ("agent_preference", "Agent favored", COLORS["agent"]),
        ("patient_preference", "Patient favored", COLORS["patient"]),
        ("tie_rate", "Tie", "#94a3b8"),
    ]:
        heights = stats[column].to_numpy()
        ax.bar(np.arange(2), heights, bottom=bottom, width=0.55, color=color, label=label)
        for x, height, base in zip(range(2), heights, bottom):
            if height > 0.04:
                ax.text(x, base + height / 2, f"{height:.1%}",
                        ha="center", va="center", color="white", weight="bold")
        bottom += heights
    ax.set_title("Test" if split == "test" else "Generalization test")
    ax.set_xticks(range(2), [
        f"{voice.capitalize()}\nn = {int(stats.loc[voice, 'sentences']):,}" for voice in VOICES
    ])
    ax.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_axisbelow(True)
    ax.grid(axis="y", color="#e2e8f0")
axes[0].set_ylabel("Share of sentences")
fig.suptitle("Baseline answer preferences", fontsize=17, weight="bold", y=0.99)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center",
           bbox_to_anchor=(0.5, 0.055), ncol=3, frameon=False)
fig.text(0.5, 0.015, "Strict pairwise noun-logit comparisons; ties favor neither answer.",
         ha="center", color="#64748b", fontsize=9)
fig.subplots_adjust(top=0.82, bottom=0.24, wspace=0.18)
save_figure(fig, "baseline_preferences")

## Change in the counterfactual answer score

Each line edits one role token at one block, using a fresh forward pass. The two role edits are separate conditions. Positive changes favor the patient more than the original prompt did. Zero is unchanged. All sentence examples contribute, including examples whose baseline already favored the patient.

In [ ]:
def plot_layers(metric, title, ylabel, filename, footnote, *, reference=False, percent=False):
    """Compare four edit conditions using common axes across split/voice panels."""
    fig, axes = plt.subplots(2, 2, figsize=(11.5, 8), sharex=True, sharey=True)
    for row_index, split in enumerate(SPLITS):
        for col_index, voice in enumerate(VOICES):
            ax = axes[row_index, col_index]
            data = summary[(summary.split == split) & (summary.voice == voice)]
            stats = baseline_stats.loc[(split, voice)]
            for means, role in CONDITION_ORDER:
                line = data[(data.mean_type == means) & (data.edited_role == role)].sort_values("layer")
                assert line.layer.tolist() == LAYERS
                ax.plot(line.layer, line[metric], color=COLORS[role],
                        linestyle=STYLES[means], linewidth=1.9)
            ax.axhline(0, color="#64748b", linewidth=0.9, zorder=0)
            if reference:
                ax.axhline(stats.mean_logit_diff, color="#334155", linestyle=":",
                           linewidth=1.5)
            split_title = "Test" if split == "test" else "Generalization test"
            denominator = (
                f"{int(stats.agent_wins):,} baseline agent wins"
                if metric == "conditional_flip_rate"
                else f"n = {int(stats.sentences):,}"
            )
            ax.set_title(f"{split_title} / {voice}  |  {denominator}", loc="left")
            ax.set_xlim(LAYERS[0], LAYERS[-1])
            ax.set_xticks([0, 4, 8, 12, 16, 20, 24, 28, 31])
            ax.grid(color="#e2e8f0", linewidth=0.65)
            ax.set_axisbelow(True)
            if percent:
                ax.set_ylim(0, 1)
                ax.yaxis.set_major_formatter(PercentFormatter(1))
            if row_index == 1:
                ax.set_xlabel("Transformer block (0-based)")
            if col_index == 0:
                ax.set_ylabel(ylabel)
    handles = [
        Line2D([0], [0], color=COLORS[role], linestyle=STYLES[means],
               linewidth=2, label=CONDITION_LABELS[(means, role)])
        for means, role in CONDITION_ORDER
    ]
    if reference:
        handles.append(Line2D([0], [0], color="#334155", linestyle=":",
                              linewidth=1.5, label="Unedited baseline"))
    fig.suptitle(title, fontsize=18, weight="bold", y=0.985)
    fig.text(0.5, 0.945, "Pythia 6.9B / agent question / intervention strength 1",
             ha="center", color="#64748b", fontsize=10)
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.065),
               ncol=2 if not reference else 3, frameon=False, fontsize=10)
    fig.text(0.5, 0.017, footnote, ha="center", color="#64748b", fontsize=9)
    fig.subplots_adjust(top=0.87, bottom=0.23, hspace=0.28, wspace=0.12)
    save_figure(fig, filename)

plot_layers(
    "mean_logit_diff_change",
    "Effect of role edits across layers",
    "Mean change in patient − agent logit",
    "logit_change",
    "Positive values shift toward the patient. All sentences contribute. Curves are descriptive means.",
)

## Scores relative to the baseline

The dotted line is the mean unedited score for that panel. Positive scores favor the patient on average; negative scores favor the agent. Averaged logit differences and preference rates answer different questions because score magnitudes vary between examples.

In [ ]:
plot_layers(
    "intervened_mean_logit_diff",
    "Answer scores after role edits",
    "Mean patient − agent logit",
    "answer_scores",
    "Above zero favors the patient; below zero favors the agent. Dotted lines show the unedited baseline.",
    reference=True,
)

## Flips among baseline agent wins

This plot conditions on examples that initially favored the agent. Its denominator is the number of baseline agent wins in the panel, shown in the title.

The saved summary's `agent_to_patient_flip_rate` divides flip counts by **all** sentences. Dividing it by `baseline_agent_preference_rate` gives the conditional rate used here. A tie after the edit is not a flip. This distinction matters because baseline agent preference is near 50% for active sentences.

In [ ]:
plot_layers(
    "conditional_flip_rate",
    "Agent-to-patient answer flips",
    "Flips / baseline agent wins",
    "conditional_flips",
    "Only strict agent-to-patient flips count. Denominator: baseline agent wins. Ties are excluded from wins.",
    percent=True,
)

## Descriptive peaks and provenance

The table selects the largest mean logit change across the entire sweep for each split and voice. These are descriptive maxima, not independently selected or validated best layers. All four conditions remain visible in the curves.

In [ ]:
peak_indices = summary.groupby(GROUPS).mean_logit_diff_change.idxmax()
peak_columns = GROUPS + [
    "layer", "mean_type", "edited_role", "mean_logit_diff_change",
    "intervened_patient_preference_rate", "conditional_flip_rate",
]
peaks = summary.loc[peak_indices, peak_columns].set_index(GROUPS)
peaks = peaks.reindex(baseline_stats.index).reset_index()
display(peaks.round(4))
peaks.to_csv(FIGURES / "descriptive_peaks.csv", index=False)
baseline_stats.to_csv(FIGURES / "baseline_statistics.csv")

provenance = {
    "run_directory": str(RUN.resolve()),
    "run_created_at": metadata["created_at"],
    "model": metadata["model"],
    "execution_model": metadata["execution_model"],
    "schema_version": metadata["schema_version"],
    "source_sha256": {
        name: hashlib.sha256((RUN / name).read_bytes()).hexdigest()
        for name in ("metadata.json", "baseline.csv", "summary.csv")
    },
    "conditional_flip_rate": "agent_to_patient_flip_rate / baseline_agent_preference_rate",
    "uncertainty": "Descriptive curves; no uncertainty intervals",
}
(FIGURES / "plot_metadata.json").write_text(json.dumps(provenance, indent=2) + "\n")
print(f"Figures and tables saved to {FIGURES}")

## Interpretation

- Baseline agent preference is approximately 48–50% for active sentences and 64–67% for passive sentences. A shift toward the patient should therefore be interpreted alongside the baseline and conditional flip curves.
- The largest mean shifts for active sentences occur at block 24 with agent edits using means for the matching voice. Passive peaks occur at block 25 on `test` and block 20 on `gen_test`, with different edit conditions.
- Block 31 has zero effect for every condition. The edit occurs at a noun token after the final transformer block, leaving no later attention layer to transmit it to the answer position.
- These plots describe this fixed prompt and vocabulary. They do not establish that role means encode pure syntax or that the descriptive peak will generalize to a different evaluation.